In [ ]:
# =============================================================================
# FUMD-AI Training & Postprocessing Workflow -- run_pipeline: local/interactive orchestrator
# =============================================================================
# Summary:      Chains Steps 1-5 (load/window -> train -> evaluate -> aggregate
#               metrics -> map migration events). Runs locally/interactively,
#               or as a single Slurm job (slurm/run_pipeline.sbatch) -- see
#               this notebook's own markdown cell below, and README's
#               "Execution environment" section, for that and the
#               alternative per-step Slurm submission pattern.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-training-postprocessing-workflow
# Version:      0.1.3
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# =============================================================================

# run_pipeline — local/interactive orchestrator

Chains Steps 1 -> 2 -> 3 -> 4 -> 5 in one call, each step's parameters set
from this notebook's own parameters cell below, using
`fumd_training_workflow.notebook_runner.run_step` (overwrite-the-
parameters-cell-then-execute -- the same trick `papermill` is built around,
implemented directly on `nbformat`/`nbclient` instead: see
`notebook_runner.py`'s module docstring for why this project doesn't take a
`papermill` dependency).

**Also submittable to Slurm as one job**, the same way this runs locally:
`slurm/run_pipeline.sbatch` chains all five steps in a single GPU
allocation (this notebook's own kernel spawns each step's kernel as a
plain OS subprocess, which inherits the job's GPU access the same way any
other child process would). That template also documents the tradeoff
against submitting steps individually (`slurm/train_model.sbatch` /
`slurm/evaluate_model.sbatch`, one notebook per job) -- worth reading
before picking either path, and see README's "Execution environment"
section for the fuller version.

Out of the box this runs Steps 1-5 against the bundled tiny example
dataset (+ its matching events slice) with a small `EPOCHS` -- replace
`DATASET_PATH` with a real preprocessing-workflow `dataset_labeled_w<W>.csv`
and raise `EPOCHS` for an actual training run.

Step 5 (map migration events, no GPU needed) is chained too, reusing
`DATASET_PATH` for vehicle positions -- set `EVENTS_CSV_PATH` to your run's
`events_all_w<W>.csv`, or to `""`/`None` to skip Step 5 entirely if you
don't have one. It still stands on its own as `jupyter execute
notebooks/step_5_map_migration_events.ipynb` if you'd rather run it
separately (e.g. with a hand-curated `BS_COORDS` overlay -- see that
notebook's own parameters cell, not exposed as a top-level parameter here).

There's also an optional, independent **combined-dataset track**: set
`COMBINE_DATASET_PATHS` (empty/off by default) to train and evaluate a
second model on several `dataset_labeled_w<W>.csv` runs combined into one
larger training set, including its own Steps 2-5 -- see the "Step 1b"
section near the bottom of this notebook for the full parameters and
what it produces.


In [ ]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # `jupyter execute` sets the kernel's cwd to this notebook's OWN
                     # directory (see nbclient's cli.py), not the repo root -- normalize
                     # back so `sys.path.insert(0, "src")` and every relative default
                     # below resolve the same way whether this notebook is run standalone
                     # (jupyter execute notebooks/step_N...ipynb, e.g. via slurm/*.sbatch)
                     # or chained by run_pipeline.ipynb (whose own cwd is already the repo
                     # root, and which additionally sets cwd explicitly -- see
                     # fumd_training_workflow.notebook_runner.run_step).

sys.path.insert(0, "src")
from fumd_training_workflow import notebook_runner as nr
from fumd_training_workflow import migration_map as mm

In [ ]:
# ---- Parameters ----
DATASET_PATH = "example-data/dataset_labeled_example.csv"
OUTPUT_DIR = "pipeline_run"
NOTEBOOKS_DIR = "notebooks"

SEQUENCE_LENGTH = 6
FUTURE_STEPS = [1, 2, 3, 4, 5, 6, 7]
VAL_FRACTION = 0.2
RANDOM_SEED = 42

EPOCHS = 5              # kept small so the bundled example trains in a couple of minutes; raise for a real run
BATCH_SIZE = 64
EARLY_STOPPING_PATIENCE = 3

# Optional: other dataset_labeled_w<W>.csv paths for Step 3's cross-run generalization check
CROSS_RUN_DATASET_PATHS = []

# Optional (default off): also combine several dataset_labeled_w<W>.csv
# runs into one larger training set (Step 1b), train a second model on it,
# and evaluate that model too -- a separate, independent track from the
# main DATASET_PATH run above, not a replacement for it. Skipped entirely
# when COMBINE_DATASET_PATHS is empty (the default). See
# notebooks/step_1b_combine_and_window_datasets.ipynb's own markdown cell
# for the full reasoning (hold out at least one run from
# COMBINE_DATASET_PATHS and list it in COMBINE_CROSS_RUN_DATASET_PATHS
# instead, the same way CROSS_RUN_DATASET_PATHS above works for the main
# run).
COMBINE_DATASET_PATHS = []
COMBINE_OUTPUT_DIR = "pipeline_run_combined"
COMBINE_ID_OFFSET = 1_000_000
COMBINE_CROSS_RUN_DATASET_PATHS = []

# Step 5's events_all_w<W>.csv -- positions are reused from DATASET_PATH above.
# Set to "" or None to skip Step 5 entirely (e.g. if you don't have an events file).
EVENTS_CSV_PATH = "example-data/events_all_example.csv"

# Step 5's optional overlays -- BS_COORDS defaults to the same known
# Alicante site coordinates Step 5's own parameters cell defaults to
# (mm.ALICANTE_BS_COORDS -- see that cell's comment for why); the street-
# map overlays are unset by default. See step_5_map_migration_events.ipynb's
# parameters cell for what each of these means.
BS_COORDS = mm.ALICANTE_BS_COORDS
OSM_GEOPACKAGE_PATH = None
OSM_PBF_PATH = None

In [ ]:
# ---- Environment variable overrides (optional) ----
# `jupyter execute` has no -p/parameter-injection flag (unlike papermill), so
# each parameter above can also be set via an environment variable of the same
# name instead of hand-editing this cell, e.g.:
#   DATASET_PATH=/abs/path/dataset_labeled_w3.csv OUTPUT_DIR=/abs/path/pipeline_run \
#   jupyter execute run_pipeline.ipynb
# See fumd_training_workflow.notebook_runner.env_override's docstring for exact
# value-parsing rules. Unset variables leave the parameters cell's own default above untouched.
from fumd_training_workflow.notebook_runner import env_override

DATASET_PATH = env_override('DATASET_PATH', DATASET_PATH)
OUTPUT_DIR = env_override('OUTPUT_DIR', OUTPUT_DIR)
NOTEBOOKS_DIR = env_override('NOTEBOOKS_DIR', NOTEBOOKS_DIR)
SEQUENCE_LENGTH = env_override('SEQUENCE_LENGTH', SEQUENCE_LENGTH)
FUTURE_STEPS = env_override('FUTURE_STEPS', FUTURE_STEPS)
VAL_FRACTION = env_override('VAL_FRACTION', VAL_FRACTION)
RANDOM_SEED = env_override('RANDOM_SEED', RANDOM_SEED)
EPOCHS = env_override('EPOCHS', EPOCHS)
BATCH_SIZE = env_override('BATCH_SIZE', BATCH_SIZE)
EARLY_STOPPING_PATIENCE = env_override('EARLY_STOPPING_PATIENCE', EARLY_STOPPING_PATIENCE)
CROSS_RUN_DATASET_PATHS = env_override('CROSS_RUN_DATASET_PATHS', CROSS_RUN_DATASET_PATHS)
COMBINE_DATASET_PATHS = env_override('COMBINE_DATASET_PATHS', COMBINE_DATASET_PATHS)
COMBINE_OUTPUT_DIR = env_override('COMBINE_OUTPUT_DIR', COMBINE_OUTPUT_DIR)
COMBINE_ID_OFFSET = env_override('COMBINE_ID_OFFSET', COMBINE_ID_OFFSET)
COMBINE_CROSS_RUN_DATASET_PATHS = env_override('COMBINE_CROSS_RUN_DATASET_PATHS', COMBINE_CROSS_RUN_DATASET_PATHS)
EVENTS_CSV_PATH = env_override('EVENTS_CSV_PATH', EVENTS_CSV_PATH)
BS_COORDS = env_override('BS_COORDS', BS_COORDS)
OSM_GEOPACKAGE_PATH = env_override('OSM_GEOPACKAGE_PATH', OSM_GEOPACKAGE_PATH)
OSM_PBF_PATH = env_override('OSM_PBF_PATH', OSM_PBF_PATH)

## Step 1 — load & window dataset

In [ ]:
nr.run_step(
    os.path.join(NOTEBOOKS_DIR, "step_1_load_and_window_dataset.ipynb"),
    os.path.join(OUTPUT_DIR, "executed", "step_1_load_and_window_dataset.ipynb"),
    parameters={
        "DATASET_PATH": DATASET_PATH,
        "FEATURE_COLUMNS": None,
        "SEQUENCE_LENGTH": SEQUENCE_LENGTH,
        "FUTURE_STEPS": FUTURE_STEPS,
        "VAL_FRACTION": VAL_FRACTION,
        "RANDOM_SEED": RANDOM_SEED,
        "OUTPUT_DIR": OUTPUT_DIR,
    },
    cwd=os.getcwd(),
)
print("Step 1 done")

## Step 2 — train model

In [ ]:
nr.run_step(
    os.path.join(NOTEBOOKS_DIR, "step_2_train_model.ipynb"),
    os.path.join(OUTPUT_DIR, "executed", "step_2_train_model.ipynb"),
    parameters={
        "INPUT_DIR": OUTPUT_DIR,
        "OUTPUT_DIR": OUTPUT_DIR,
        "EPOCHS": EPOCHS,
        "BATCH_SIZE": BATCH_SIZE,
        "EARLY_STOPPING_PATIENCE": EARLY_STOPPING_PATIENCE,
        "RANDOM_SEED": RANDOM_SEED,
    },
    cwd=os.getcwd(),
)
print("Step 2 done")

## Step 3 — evaluate model

In [ ]:
nr.run_step(
    os.path.join(NOTEBOOKS_DIR, "step_3_evaluate_model.ipynb"),
    os.path.join(OUTPUT_DIR, "executed", "step_3_evaluate_model.ipynb"),
    parameters={
        "RUN_DIR": OUTPUT_DIR,
        "METRICS_OUTPUT_DIR": OUTPUT_DIR,
        "CROSS_RUN_DATASET_PATHS": CROSS_RUN_DATASET_PATHS,
    },
    cwd=os.getcwd(),
)
print("Step 3 done")

## Step 4 — aggregate training metrics

In [ ]:
nr.run_step(
    os.path.join(NOTEBOOKS_DIR, "step_4_aggregate_training_metrics.ipynb"),
    os.path.join(OUTPUT_DIR, "executed", "step_4_aggregate_training_metrics.ipynb"),
    parameters={
        "RUN_DIRS": [OUTPUT_DIR],
        "VARIANT_LABELS": None,
        "COMPARISON_OUTPUT_DIR": os.path.join(OUTPUT_DIR, "metrics_comparison"),
    },
    cwd=os.getcwd(),
)
print("Step 4 done")

## Step 5 — map migration events (optional)


In [ ]:
if EVENTS_CSV_PATH:
    nr.run_step(
        os.path.join(NOTEBOOKS_DIR, "step_5_map_migration_events.ipynb"),
        os.path.join(OUTPUT_DIR, "executed", "step_5_map_migration_events.ipynb"),
        parameters={
            "POSITIONS_CSV_PATH": DATASET_PATH,
            "EVENTS_CSV_PATH": EVENTS_CSV_PATH,
            "OUTPUT_DIR": os.path.join(OUTPUT_DIR, "migration_maps"),
            "BS_COORDS": BS_COORDS,
            "OSM_GEOPACKAGE_PATH": OSM_GEOPACKAGE_PATH,
            "OSM_PBF_PATH": OSM_PBF_PATH,
        },
        cwd=os.getcwd(),
    )
    print("Step 5 done")
else:
    print("Step 5 skipped -- EVENTS_CSV_PATH not set")

print(f"pipeline complete -- see {OUTPUT_DIR}/ for the trained model, metrics, and comparison plots")


## Step 1b (optional) — combine multiple datasets & train/evaluate a second model

Independent of Steps 1-5 above -- skipped entirely when `COMBINE_DATASET_PATHS`
is empty (the default). When set, runs `step_1b_combine_and_window_datasets.ipynb`
(combine + window several runs, shifting vehicle ids so they can't collide),
then Step 2 (train) and Step 3 (evaluate -- on its own held-out split always,
plus against `COMBINE_CROSS_RUN_DATASET_PATHS` if also set), then Step 4
(aggregate/compare those metrics) -- all under `COMBINE_OUTPUT_DIR`, entirely
separate from the main run's `OUTPUT_DIR`.

Step 5 also runs for this track, but differently from the main run above:
there's no single coherent "combined" positions/events dataset to map (the
three source runs' vehicle ids were only shifted to avoid training
collisions, not merged into one consistent simulation), so instead this
produces one migration-events map *per source run* in
`COMBINE_DATASET_PATHS` -- reusing the same `BS_COORDS`/`OSM_GEOPACKAGE_PATH`/
`OSM_PBF_PATH` overlays as the main run's Step 5, and each source run's own
`events_all_w<W>.csv` (derived automatically from its `dataset_labeled_w<W>.csv`
path via `migration_map.derive_events_path` -- same directory, same window
size, the convention every FUMD-AI preprocessing run follows). A source run
missing its events file has its map skipped (printed, not a hard failure)
rather than aborting the rest of the pipeline.


In [ ]:
if COMBINE_DATASET_PATHS:
    nr.run_step(
        os.path.join(NOTEBOOKS_DIR, "step_1b_combine_and_window_datasets.ipynb"),
        os.path.join(COMBINE_OUTPUT_DIR, "executed", "step_1b_combine_and_window_datasets.ipynb"),
        parameters={
            "DATASET_PATHS": COMBINE_DATASET_PATHS,
            "ID_OFFSET": COMBINE_ID_OFFSET,
            "FEATURE_COLUMNS": None,
            "SEQUENCE_LENGTH": SEQUENCE_LENGTH,
            "FUTURE_STEPS": FUTURE_STEPS,
            "VAL_FRACTION": VAL_FRACTION,
            "RANDOM_SEED": RANDOM_SEED,
            "OUTPUT_DIR": COMBINE_OUTPUT_DIR,
        },
        cwd=os.getcwd(),
    )
    print("Step 1b done")

    nr.run_step(
        os.path.join(NOTEBOOKS_DIR, "step_2_train_model.ipynb"),
        os.path.join(COMBINE_OUTPUT_DIR, "executed", "step_2_train_model.ipynb"),
        parameters={
            "INPUT_DIR": COMBINE_OUTPUT_DIR,
            "OUTPUT_DIR": COMBINE_OUTPUT_DIR,
            "EPOCHS": EPOCHS,
            "BATCH_SIZE": BATCH_SIZE,
            "EARLY_STOPPING_PATIENCE": EARLY_STOPPING_PATIENCE,
            "RANDOM_SEED": RANDOM_SEED,
        },
        cwd=os.getcwd(),
    )
    print("Step 2 (combined) done")

    # Step 3 always evaluates the combined model's own held-out split
    # (run_label="train_val") -- CROSS_RUN_DATASET_PATHS only controls
    # whether it *also* checks generalization against other runs. Making
    # this unconditional on COMBINE_CROSS_RUN_DATASET_PATHS (fixed here --
    # it used to skip Step 3, and therefore the combined model's own
    # metrics, entirely whenever no cross-run paths were given) mirrors how
    # the main track's Step 3 call above always runs, and is what Step 4
    # right after it needs: without it there'd be nothing to aggregate.
    nr.run_step(
        os.path.join(NOTEBOOKS_DIR, "step_3_evaluate_model.ipynb"),
        os.path.join(COMBINE_OUTPUT_DIR, "executed", "step_3_evaluate_model.ipynb"),
        parameters={
            "RUN_DIR": COMBINE_OUTPUT_DIR,
            "METRICS_OUTPUT_DIR": COMBINE_OUTPUT_DIR,
            "CROSS_RUN_DATASET_PATHS": COMBINE_CROSS_RUN_DATASET_PATHS,
        },
        cwd=os.getcwd(),
    )
    print("Step 3 (combined) done")

    nr.run_step(
        os.path.join(NOTEBOOKS_DIR, "step_4_aggregate_training_metrics.ipynb"),
        os.path.join(COMBINE_OUTPUT_DIR, "executed", "step_4_aggregate_training_metrics.ipynb"),
        parameters={
            "RUN_DIRS": [COMBINE_OUTPUT_DIR],
            "VARIANT_LABELS": None,
            "COMPARISON_OUTPUT_DIR": os.path.join(COMBINE_OUTPUT_DIR, "metrics_comparison"),
        },
        cwd=os.getcwd(),
    )
    print("Step 4 (combined) done")

    # Step 5 (combined track): one map per SOURCE run that went into the
    # combined training set -- see the markdown cell above for why this is
    # per-source rather than one "combined" map. Uses the same BS_COORDS/
    # OSM_GEOPACKAGE_PATH/OSM_PBF_PATH overlays as the main track's Step 5.
    for dataset_path in COMBINE_DATASET_PATHS:
        # Same run_label convention Step 3 uses for cross-run metrics
        # filenames (basename of the containing directory, e.g. "900_1") --
        # every source run shares the same dataset_labeled_w<W>.csv
        # filename, so the containing directory is what actually
        # distinguishes them.
        run_label = os.path.basename(os.path.dirname(os.path.abspath(dataset_path))) \
            or os.path.splitext(os.path.basename(dataset_path))[0]
        try:
            events_path = mm.derive_events_path(dataset_path)
        except ValueError as exc:
            print(f"Step 5 (combined, {run_label}) skipped -- {exc}")
            continue
        if not os.path.exists(events_path):
            print(f"Step 5 (combined, {run_label}) skipped -- no events file at {events_path}")
            continue
        nr.run_step(
            os.path.join(NOTEBOOKS_DIR, "step_5_map_migration_events.ipynb"),
            os.path.join(COMBINE_OUTPUT_DIR, "executed", f"step_5_map_migration_events_{run_label}.ipynb"),
            parameters={
                "POSITIONS_CSV_PATH": dataset_path,
                "EVENTS_CSV_PATH": events_path,
                "OUTPUT_DIR": os.path.join(COMBINE_OUTPUT_DIR, "migration_maps", run_label),
                "BS_COORDS": BS_COORDS,
                "OSM_GEOPACKAGE_PATH": OSM_GEOPACKAGE_PATH,
                "OSM_PBF_PATH": OSM_PBF_PATH,
            },
            cwd=os.getcwd(),
        )
        print(f"Step 5 (combined, {run_label}) done")

    print(f"combined-dataset track complete -- see {COMBINE_OUTPUT_DIR}/")
else:
    print("Step 1b/combined-dataset track skipped -- COMBINE_DATASET_PATHS not set")
